# Deploy OpenClaw with Qwen 3.5 and vLLM on AMD GPUs

**Author**: Rishi Sinha  
**Knowledge level**: Beginner

**Workshop Goal** : Deploy and personalize a fully local AI agent on AMD GPUs

**What you'll do**: 

* Deploy [Qwen3.5-122B-A10B-FP8](https://huggingface.co/Qwen/Qwen3.5-122B-A10B-FP8) with vLLM on an MI300X GPU.
*  Connect [OpenClaw](https://openclaw.ai/) to the running model and use it as a personal AI agent with **direct access to your local filesystem**.
* Customize the agent's personality and behavior by editing its Markdown configuration files 
* Build a morning brief generator for your favorite githubs and news sites 

**What you'll learn**: 

* How to Provision GPU resources through the AMD Developer Cloud 
* Serving large MOE (mixture-of-experts) models locally with inference engine
* How Markdown configuration gives you fine-grained control over agent behavior
* A method for building automated agents to solve domain-specific problems

## Background
Most AI assistants rely on the cloud, which means your prompts and data leave your machine. This tutorial shows how you can run everything locally on AMD hardware. [OpenClaw](https://openclaw.ai/) is a **local-first**, self-hosted AI agent that runs entirely on your hardware. It can read your files, run commands, and automate tasks, without the use of the cloud. You control which skills, tools, and integrations are enabled, making it a flexible foundation for personal automation with full privacy.

To power the agent, you will use Qwen3.5, a 122 billion param mixture-of-experts model which is especially good for reasoning and coding tasks. You will learn to serve the model through vLLM, a high-throughput inference engine with native ROCm support. Because MOE models activate only a subset of parameters for a token, this large model will comfortably fit on a single MI300X GPU. 


This tutorial is based on the AMD technical article [OpenCLaw on AMD Developer Cloud: Free Deployment with Qwen 3.5 and SGLang](https://www.amd.com/en/developer/resources/technical-articles/2026/openclaw-on-amd-developer-cloud-qwen-3-5-and-sglang.html).


### Table of contents

1. [Prerequisites](#prerequisites)
2. [AMD Developer Cloud credits](#amd-developer-cloud-credits)
3. [Install and launch Jupyter](#1-install-and-launch-jupyter)
4. [Launch vLLM](#2-launch-vllm)
5. [Install and configure OpenClaw](#3-install-and-configure-openclaw)
6. [Customizing your OpenClaw Agent](#4-customizing-your-openclaw-agent)
7. [Project: Building your own Morning Brief](#5-project-building-your-own-morning-brief)
8. [Conclusion](#conclusion)
9. [Cleanup](#cleanup)
10. [Further reading](#further-reading)

Let's get started!

## Prerequisites

This tutorial was developed and tested using the following setup.

### Operating system

* **Ubuntu 22.04**: Ensure your system is running Ubuntu 22.04.

### Hardware

* **AMD Instinct™ GPUs**: This tutorial was tested on a single AMD Instinct MI300X GPU (192 GB HBM3). Ensure you are using an AMD Instinct GPU or compatible hardware with ROCm support and that your system meets the [official requirements](https://rocm.docs.amd.com/projects/install-on-linux/en/latest/reference/system-requirements.html).
* We recommend using the AMD Developer Cloud (in the cell below) to get access to these GPUs!

### Software

* **ROCm 7.0**: Install and verify ROCm by following the [ROCm install guide](https://rocm.docs.amd.com/projects/install-on-linux/en/latest/install/quick-start.html). After installation, confirm your setup using:

    ``` bash
    amd-smi
    ```

    This command lists your AMD GPUs with relevant details.

    **Note**: For ROCm 6.4 and earlier, use the `rocm-smi` command instead.

* **Docker**: Ensure Docker is installed and configured correctly. Follow the Docker installation guide for your operating system.

   **Note**: Ensure the Docker permissions are correctly configured. To configure permissions to allow non-root access, run the following commands:

   ``` bash
   sudo usermod -aG docker $USER
   newgrp docker
   ```

   Verify Docker is working correctly:

   ``` bash
   docker run hello-world
   ```

### AMD Developer Cloud credits

**Skip this cell if you have already gotten access to a droplet or have a local GPU**

The [AMD AI Developer Program](https://www.amd.com/en/developer/ai-dev-program.html) provides members with $100 of free AMD Developer Cloud credits, which is enough for approximately 50 hours of usage.

To get started:

1. **Sign up** for the [AMD AI Developer Program](https://www.amd.com/en/developer/ai-dev-program.html). Existing AMD account holders can sign in and enroll directly.
2. **Activate your credits** from the AMD AI Developer Portal profile page.
3. **Create a GPU Droplet**: Select a single MI300X instance with the ROCm Software image, and add your SSH key.

Once the droplet is created, connect via SSH:

``` bash
ssh root@<YOUR_DROPLET_IP>
```

Members who showcase useful applications and publicly share their projects may also apply for additional credits.

## Let's get Started

### 1. Install and launch Jupyter

In the droplet terminal, run these commands

``` bash
python3 -m venv .venv
source .venv/bin/activate
pip install jupyterlab
```

After those commands run, start the Jupyter server in the same terminal:

``` bash
jupyter-lab --ip=0.0.0.0 --port=8888 --no-browser --allow-root
```

After running the jupyter-lab command, you should be able to click on a link in the terminal to access the notebook. It should appear like: 

`http://127.0.0.1:<PORT>/lab?token=<TOKEN_VALUE>`


**Note**: Ensure port `8888` is not already in use on your system before running the above command. If it is, you can specify a different port by replacing `--port=8888` with another port number, for example, `--port=8890`.

**Note**: Ensure the notebook file is either copied to the `/workspace` directory or uploaded into the Jupyter Notebook environment after it starts. You can download this notebook from the [AI Developer Hub GitHub repository](https://github.com/ROCm/gpuaidev).


**Note**: The following steps should be executed within your Jupyter notebook after successfully launching the Jupyter server.

### 2. Launch vLLM 

Running the cell below starts a Docker container to serve the [Qwen3.5-122B-A10B-FP8](https://huggingface.co/Qwen/Qwen3.5-122B-A10B-FP8) model with the latest vLLM. Be prepared that it can take a few minutes to load the model when you run it for the first time.

**Note**: Replace `abc-123` in the command below with a secure, unique API key.

In [ ]:
%%bash
docker run -d \
    --name vllm_server \
    --ipc=host \
    --privileged \
    --device=/dev/kfd \
    --device=/dev/dri \
    -p 8000:8000 \
    vllm/vllm-openai-rocm:latest \
    --model Qwen/Qwen3.5-122B-A10B-FP8 \
    --served-model-name qwen3-5-122b \
    --host 0.0.0.0 --port 8000 \
    --api-key abc-123 \
    --reasoning-parser qwen3 \
    --enable-auto-tool-choice \
    --tool-call-parser qwen3_coder \
    --trust-remote-code

Verify the server is running by running the cell below. It might take a few minutes for the model to load.

You can also monitor the loading progress in your terminal with:

```bash
docker logs -f vllm_server
```

Look for `Uvicorn running on http://0.0.0.0:8000` to know it's ready.

In [ ]:
import urllib.request, json, subprocess, time, pathlib, sys, os, shutil

# Checks for server to be ready (polls up to 5 minutes)
print("Waiting for server to be ready", end="", flush=True)
deadline = time.time() + 300
ready = False
while time.time() < deadline:
    try:
        req = urllib.request.Request(
            "http://localhost:8000/v1/models",
            headers={"Authorization": "Bearer abc-123"}
        )
        with urllib.request.urlopen(req, timeout=3) as r:
            models = json.loads(r.read())
        ready = True
        break
    except Exception:
        print(".", end="", flush=True)
        time.sleep(5)

if ready:
    print("\n✅ Server is ready")
    for m in models.get("data", []):
        print(f"   Model: {m['id']}")
else:
    print("\n❌ Server did not become ready within 5 minutes")

### Troubleshooting

If the setup script fails, check these common issues:

- **"Container name already in use"**: A previous run left a container with the same name. Remove it and retry:
  ``` bash
  docker rm -f vllm_server
  ```
- **Out of memory (OOM)**: The model requires approximately 80 GB of GPU memory. Verify your GPU has enough free memory:
  ``` bash
  amd-smi monitor
  ```
- **Health check times out**: Inspect the container logs for errors:
  ``` bash
  docker logs vllm_server
  ```

### 3. Install and configure OpenClaw

#### The Overview

Before we start OpenClaw, here's an overview of what we will touch:

| Concept | What it is | Where it lives |
|---|---|---|
| **Gateway** | Connects OpenClaw to the model running on AMD hardware | Process on port 18789 |
| **Workspace** | The agent's "brain" — files it reads on every message | `~/.openclaw/workspace/` |
| **Tools** | How the agent acts: read files, run shell commands, write edits | Defined in `TOOLS.md` |
| **Skills** | Saved, reusable workflows the agent can follow on demand | `workspace/skills/` |

#### 🖥️ Open a new terminal

Open a new terminal in JupyterLab by clicking the `+` button to open a new launcher tab, then find the **terminal icon** (not a notebook cell). The following steps must run in a terminal because the OpenClaw gateway needs to persist as a background process.

#### Install OpenClaw

Run in your terminal:

```bash
curl -fsSL https://openclaw.ai/install.sh | bash
```

#### Onboard your OpenClaw

Run in your terminal:

```bash
openclaw onboard
```

OpenClaw will walk you through setup. **Choose these settings:**

1. **Security Continue?** → `Yes`
2. **Setup mode** → `QuickStart`
3. **Model/auth provider** → `vLLM`
4. **vLLM base URL** → Leave as `http://127.0.0.1:8000/v1`
5. **vLLM API key** → `abc-123`
6. **vLLM model** → `qwen3-5-122b`
7. **Default model** → `Keep current (vllm/qwen3-5-122b)`
8. **Select channel** → `Skip for now`
9. **Search provider** → `Skip for now`
10. **Configure skills now?** → `No`
11. **Enable hooks?** → `Skip for now`
12. **How do you want to hatch your bot?** → `Hatch in Terminal (recommended)`

The OpenClaw interface will appear and once it says `gateway connected`, it's ready! This is where you will chat with your OpenClaw agent whenever you see the 🦞 emoji.

You can now start chatting with your OpenClaw agent powered by Qwen3.5 on AMD hardware.



#### 🦞 Say hello to your agent

Send your first message! Try something like:

> *"Hey! Who are you and what can you do?"*

The agent will introduce itself and tell you about its capabilities!

### 4. Understanding your OpenClaw Agent's Configuration

OpenClaw's behavior is defined by plain Markdown files in its workspace. These files are read on every message, giving the agent its personality, policies, and context about you.

```
~/.openclaw/workspace/
├── SOUL.md       ← WHO the agent is: values, personality, tone
├── AGENTS.md     ← HOW it operates: startup rules, memory, red lines
├── IDENTITY.md   ← WHAT others see: name, emoji, public metadata
├── USER.md       ← WHO you are: context the agent reads about you
├── TOOLS.md      ← local environment notes (SSH hosts, device names, etc.)
├── MEMORY.md     ← long-term curated memory across sessions
├── HEARTBEAT.md  ← checklist for periodic background checks
├── BOOTSTRAP.md  ← first-run ritual (deleted after onboarding)
└── memory/       ← daily session logs (today + yesterday auto-loaded)
```

**SOUL.md is personality. AGENTS.md is policy.** Every message the agent receives, it re-reads both.

#### 🦞 Your turn!

In your OpenClaw terminal, ask the agent about its own configuration:

- *"List all your MD files"*
- *"Show me what's in your SOUL.md, IDENTITY.md and BOOTSTRAP.md files"*
- *"What do you know about me?"*
- *"What tools do you have access to?"*

The agent will read its own workspace files and explain what controls its personality, identity, and behavior.

### 5. Project: Building your own Morning Brief

**The Problem**

1. As AI/ML developers, we have to stay on top of so many different repos

eg. 
* vLLM
* OpenClaw
* ROCm (our software stack for running programs on AMD GPUs that you should checkout)
* Transformers
* SGLang
etc... 

**The Solution** 

In this project you'll build an AI agent that:

1. Checks your favorite repos for updates
2. Filters based on your interests
3. Generates a personalized brief
4. Runs automatically every morning

Let's build it with OpenClaw!


---

#### Part 1: Schedule with Cron

Instead of writing a script, we'll schedule an AI agent to check our repos every morning.

**Why an agent instead of a static script?**
- Adapts if something breaks
- Improves over time
- Summarizes in natural language
- Handles edge cases intelligently

Cron jobs have been around since the start of Unix. They run a task on a repeating schedule. 

The difference is with OpenClaw, you create cron jobs through natural language that can spawn agents. No code or complicated commands required.


##### Step 1: Create a Daily Cron Job

Instead of writing a script, we'll schedule an AI agent to check our repos every morning. Feel free to change the repos or instructions to match your interests.

#### 🦞 Ask your OpenClaw agent

> *"Generate a morning brief for me every morning at 8 AM. Check repos sgl-project/sglang, vllm-project/vllm, huggingface/transformers, ROCm/ROCm, openclaw/openclaw for performance, GPU, and breaking changes. Report on big PRs or details. Also, give me a summary of the latest news"*

##### Step 2: Verify the Job

Check that the job was created by opening the **Cron Jobs** tab on the dashboard, or by running this command in your terminal:

#### 🖥️ Run in your terminal

```bash
openclaw cron list
```


*Note: if you see a delivery error, that's because we haven't configured a chat channel — it can be safely ignored.*

##### Step 3: Test It Now

The cron job is scheduled for 8 AM UTC, but let's trigger it right now to see the agent in action.

#### 🦞 Ask your OpenClaw agent

> *"Can you run the cron job right now and give me the output?"*

#### Part 1 Complete!

**What you built**:
- A cron job that runs an AI agent every morning at 8 AM UTC
- A personalized brief covering your favorite repos and news

**What you learned**:
- How to schedule agents with natural language
- How to trigger and monitor cron jobs

---

Next, let's teach the agent to remember your preferences.


---

#### Part 2: Teaching the Agent to Remember

The brief runs every day, but right now each run starts fresh. What if you want it to skip CI noise, focus on GPU changes, or group results differently? You could re-type those instructions every time, or you could teach the agent to **remember**.

##### How OpenClaw's Memory Works

OpenClaw has a file-based memory system that persists across sessions:

- **Daily notes** (`memory/YYYY-MM-DD.md`) : the agent logs conversations, events, and decisions each day.
- **Long-term memory** (`MEMORY.md`) : important context and preferences that carry forward indefinitely.
- **Workspace files** (`~/.openclaw/workspace/`) : the agent can read and write files here. Your morning briefs are saved here too, so it can compare today's brief to yesterday's.

When you use the keyword **"remember"**, the agent writes your preferences into its memory files. The next cron run reads those files and applies them automatically.


##### Step 1: Tell the Agent to Remember Your Preferences

By using the keyword **"remember"**, the agent will save your preferences so they persist across sessions and future cron runs.

#### 🦞 Ask your OpenClaw agent

> *"Remember: for the morning brief, include only PRs and CI/infrastructure changes. Focus on contributors and performance over the last 90 days. Group results by repo."*

##### Step 2: Run the Brief and See the Changes

Trigger the morning brief again. The agent reads your saved preferences and applies them to the output.

#### 🦞 Ask your OpenClaw agent

> *"Run the morning brief now and show me the result."*

##### Step 3: Keep Refining

Send more "remember" messages anytime to update your preferences:

- *"Remember to also include trending HuggingFace models"*
- *"Remember to focus more on ROCm compatibility issues"*

Each message updates the agent's memory files, so the next cron run automatically picks up your latest preferences.


#### 🦞 See what the agent remembers

As a final step, ask the agent to show you its memory file. This is where all the preferences you taught it are stored:

> *"Show me your MEMORY.md file"*

You should see the preferences you set earlier — like skipping documentation-only PRs and grouping results by repo. This is the file-based memory system that carries forward across sessions and cron runs, tying back to the workspace files you explored in [Section 4](#4-understanding-your-openclaw-agents-configuration).

#### Part 2 Complete!

**What you built**:
- Persistent preferences that carry across sessions and cron runs
- A feedback loop driven entirely by natural language

**What you learned**:
- How OpenClaw's memory system (daily notes, MEMORY.md, workspace files) works
- How the "remember" keyword saves preferences to memory
- How to refine agent behavior without editing config files

---

**Want to go further?** Check out the bonus ideas below.


---

#### Bonus: Advanced Features

Here are some ideas to take your morning brief further.

##### Trend Detection

Compare today's brief against previous ones to spot trends. (Needs a few days of saved briefs first.)

#### 🦞 Ask your OpenClaw agent

> *"How do today's trends compare to the last 10 morning briefs?"*

##### Webhook Delivery

Send the brief to Slack, Discord, or Telegram (might need some configuration).

#### 🦞 Ask your OpenClaw agent

> *"Send my morning briefs to Slack at this webhook: https://hooks.slack.com/services/YOUR/WEBHOOK/URL"*

##### Multi-Agent Orchestration

Spawn separate agents for each repo for parallel processing.

#### 🦞 Ask your OpenClaw agent

> *"For the morning brief, spawn a separate agent for each repo. Each one fetches and summarizes its repo, then aggregate all results."*

*Note: this can take a while.*

---

#### Workshop Complete!

##### What You Built

- **Natural-Language Scheduling** — Created cron jobs by talking to the agent
- **Automated Morning Briefs** — Runs every morning without manual effort
- **Persistent Memory** — Taught the agent to remember your preferences across sessions
- **Conversational Refinement** — Tuned the output through messages, not config files


## Conclusion

In this tutorial, you deployed **Qwen3.5-122B-A10B-FP8** on a single AMD Instinct MI300X GPU using vLLM, and connected it to **OpenClaw** as a fully local AI agent.

You then customized your agent, and then used it to solve a problem that developers and engineers face everyday. This should be an illustration into the power of local agents, that can be powered locally through the AMD Developer Cloud! 

See the [Further reading](#further-reading) section for deeper dives.

## Cleanup

When you're done, stop and remove the vLLM container:

In [ ]:
!docker stop vllm_server && docker rm vllm_server

## Further reading

- [OpenCLaw on AMD Developer Cloud: Free Deployment with Qwen 3.5 and SGLang](https://www.amd.com/en/developer/resources/technical-articles/2026/openclaw-on-amd-developer-cloud-qwen-3-5-and-sglang.html) — the original AMD technical article this tutorial is based on.
- [OpenClaw](https://openclaw.ai/) — official page for OpenClaw.
- [vLLM](https://github.com/vllm-project/vllm) — the inference framework used in this tutorial.
- [Qwen3.5-122B-A10B-FP8 on Hugging Face](https://huggingface.co/Qwen/Qwen3.5-122B-A10B-FP8) — the model card.
- [AMD AI Developer Program](https://www.amd.com/en/developer/ai-dev-program.html) — sign up for free AMD Developer Cloud credits.
- [AMD Infinity Hub](https://www.amd.com/en/developer/resources/infinity-hub.html#f-amd_hub_category=AI%20%26%20ML%20Models) — ready-made Docker images for AI with ROCm.